<a href="https://colab.research.google.com/github/emmanuelmassawe/fine-tunning-llm-with-lora-and-qlora/blob/main/fine_tunning_llm_with_lora_and_qlora_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#this is llm fine-tuning using lora and qlora

##install necessary packages

In [2]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate

##import neccesary libraries

In [3]:
import gc
import os

import torch
from datasets import load_dataset
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

In [4]:
## CONFIG: badilisha hapa tu
MODE = "qlora"                       # "lora" (16-bit) au "qlora" (4-bit)
MODEL_NAME = "Qwen/Qwen2.5-0.5B"     # model ndogo kwa majaribio
DATASET = "yahma/alpaca-cleaned"     # dataset ya Hugging Face
DATA_FILE = None                     # mfano: "data.json" ukitumia faili lako (inapita DATASET)
MAX_SAMPLES = 1000
MAX_LENGTH = 512
EPOCHS = 1
LR = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MERGE_AFTER_TRAINING = False

In [5]:
# Hifadhi kwenye Drive ikiwa imeunganishwa, vinginevyo kwenye /content
_base = "/content/drive/MyDrive" if os.path.isdir("/content/drive/MyDrive") else "/content"
OUTPUT_DIR = f"{_base}/out-{MODE}"

In [6]:
# T4 haina bf16, kwa hiyo tunachagua kiotomatiki
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    DTYPE = torch.bfloat16
else:
    DTYPE = torch.float16

In [7]:
# ---------------------------------------------------------------
# HATUA 1: Pakia model (hapa ndipo quantization inapotokea)
# ---------------------------------------------------------------
def load_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if MODE == "qlora":
        # Uzito wa model unahifadhiwa kwa biti 4
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",         # NormalFloat4
            bnb_4bit_use_double_quant=True,    # quantize hata constants
            bnb_4bit_compute_dtype=DTYPE,      # hesabu kwa 16-bit
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb_config, device_map="auto"
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype=DTYPE, device_map="auto"
        )
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()

    model.config.use_cache = False
    return model, tokenizer
